# Epigraphy field assistant

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/ghonchu/blob/main/notebooks/06_epigraphy_field_assistant.ipynb)

For historians and archaeologists: point the camera at an inscription, a coin or a sherd, and the assistant names the likely script family and what it can read. Raking light helps.

**Initial drop, 9 November 2026. Expect updates soon.** Extracted from the design discussion of the Ghonchu course (8 November 2026) and not yet run end to end; what was checked and what was changed is listed in [origin/SEED-DIFF.md](https://github.com/zistgah/ghonchu/blob/main/origin/SEED-DIFF.md).

Press ▶ on each step in order. Each step's code is folded behind its title; double-click a title to see it.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Licensed under GPL-3.0-or-later. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending).

**Privacy and safety.** The camera picture travels from your phone's browser to this Colab runtime, which runs on Google's cloud, not on your phone; nothing is kept unless a cell saves it. Point the camera at the robot or the object, never at a person without their informed consent; children use it with an adult, and a parent's consent comes first. Do not show patient, client or personal information. These assistants teach; they are not medical, psychological or diagnostic devices, and no prompt makes anything compliant with GDPR, HIPAA or any other law.

Built from the shared vision tutor with this domain's prompt, question, panel, voice and timing exactly as discussed (14:32); the received cell differed from the shared one only in those, in its function names and in two panel colours, all recorded in origin/received.

In [17]:
# 1. Install required acceleration libraries for running 4-bit Vision Models on T4 with compatible versions
!pip install -q "unsloth[colab-new]" --upgrade
!pip install -q --upgrade xformers bitsandbytes accelerate "transformers>=4.56.2,<4.57.0" "trl>=0.18.2,<=1.15.0" requests
!pip install -q misty-doi

# Import unsloth first to ensure all memory-efficiency patches are fully initialized before anything else
from unsloth import FastVisionModel

import gc
import os
import torch
import requests
import urllib.parse
import datetime
import subprocess

# --- Optional Google Drive Caching Option (Perfect for GitHub Sharing!) ---
USE_DRIVE_CACHE = True #@param {type:"boolean"}
CACHE_DIR = None
LOGS_DIR = None

if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        print("Mounting Google Drive for caching...")
        drive.mount('/content/drive')
        CACHE_DIR = "/content/drive/MyDrive/ghonchu_model_cache"
        LOGS_DIR = "/content/drive/MyDrive/ghonchu_ots_logs"
        os.makedirs(CACHE_DIR, exist_ok=True)
        os.makedirs(LOGS_DIR, exist_ok=True)
        print(f"Using Google Drive cache directory: {CACHE_DIR}")
        print(f"Using Google Drive OTS logs directory: {LOGS_DIR}")
    except Exception as e:
        print(f"Could not mount Google Drive ({e}). Falling back to local temporary storage.")
        CACHE_DIR = None
        LOGS_DIR = "/tmp/ghonchu_ots_logs"
        os.makedirs(LOGS_DIR, exist_ok=True)
else:
    LOGS_DIR = "/tmp/ghonchu_ots_logs"
    os.makedirs(LOGS_DIR, exist_ok=True)

# Free any previously occupied GPU memory from crashed runs
if torch.cuda.is_available():
    if 'model' in globals():
        del model
    if 'tokenizer' in globals():
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

import base64
import numpy as np
import cv2
from io import BytesIO
from PIL import Image
from IPython.display import display, Javascript
import google.colab.output
import IPython

# 2. Load the Vision-Language Model in 4-bit
print("Loading Ghonchu Vision Brain (Qwen2.5-VL-7B 4-bit)...")
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True,
    cache_dir = CACHE_DIR
)
FastVisionModel.for_inference(model) # Optimize model hooks for fast generation loops

# Load OpenCV Haar Cascade for Face Detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 3. Define the Global Pedagogical System Prompt for your Robotics Course
FIELD_SYSTEM_PROMPT = "\nYou are Ghonchu, an expert Epigraphist, Palaeographer, and Field Anthropologist.\nYou are inspecting an ancient inscription, rock edict, coin, or cultural artifact through a mobile camera frame.\nCarefully examine the visual data provided:\n1. Identify the script lineage or language family if characters are present (e.g., Brahmi, Kharosthi, Grantha, Vatteluttu, etc.).\n2. Attempt to transliterate visible characters, recognize iconographic patterns, or note physical weathering states.\n3. Keep your output to ONE clear, descriptive sentence that can be easily spoken out loud by the browser's audio synthesis engine.\nDo not speculate wildly; if the script is too weathered or lighting is poor, explicitly state so.\n"

def log_and_stamp_artifact(prompt, response, pil_image=None):
    """Saves conversation logs and images to GDrive, then notarizes them with OpenTimestamps via misty-doi."""
    timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = os.path.join(LOGS_DIR, f"log_{timestamp_str}.txt")
    img_filename = os.path.join(LOGS_DIR, f"image_{timestamp_str}.jpg")

    # Write Transcript
    with open(log_filename, "w", encoding="utf-8") as f:
        f.write(f"Timestamp: {datetime.datetime.now().isoformat()}\n")
        f.write(f"User Prompt: {prompt}\n")
        f.write(f"Ghonchu Output: {response}\n")

    # Save Image if provided
    if pil_image is not None:
        pil_image.save(img_filename, "JPEG")

    # Stamp with misty-doi OTS CLI
    try:
        subprocess.run(["misty", "ots", "stamp", log_filename], capture_output=True, text=True)
        if pil_image is not None:
            subprocess.run(["misty", "ots", "stamp", img_filename], capture_output=True, text=True)
        print(f"[OTS Active] Saved log and notarized snapshot to Google Drive.")
    except Exception as e:
        print(f"[OTS Warning] OpenTimestamps stamp execution bypassed or errored: {e}")

def start_ghonchu_eye_tutor():
    js_code = Javascript('''
        async function runGhonchuTutor() {
            const container = document.createElement('div');
            container.style = 'padding: 16px; background: #1c1917; border-radius: 12px; color: #f5f5f4; font-family: sans-serif; max-width: 480px; margin: 12px auto; border: 2px solid #d97706; text-align: center; position: relative;';

            const title = document.createElement('h3');
            title.textContent = '🏛️ Ghonchu Field Epigraphy Assistant';
            title.style = 'color: #d97706; margin-top: 0; font-size: 18px;';

            // Ghonchu mascot container overlay
            const mascotContainer = document.createElement('div');
            mascotContainer.style = 'width: 120px; height: 120px; margin: 10px auto; border-radius: 50%; overflow: hidden; border: 3px solid #d97706; background: #000; display: flex; justify-content: center; align-items: center; position: relative;';

            const mascotImg = document.createElement('img');
            mascotImg.src = 'https://zisgah.org/assets/images/ghonchu.png';
            mascotImg.style = 'width: 100%; height: 100%; object-fit: cover;';
            mascotImg.onerror = () => {
                mascotImg.src = 'https://img.icons8.com/emoji/96/smiling-face.png';
            };
            mascotContainer.appendChild(mascotImg);

            const style = document.createElement('style');
            style.innerHTML = `
                @keyframes talking {
                    0% { transform: scale(1); border-color: #d97706; }
                    50% { transform: scale(1.1); border-color: #f59e0b; }
                    100% { transform: scale(1); border-color: #d97706; }
                }
                .talking-active { animation: talking 0.4s infinite; }
            `;
            document.head.appendChild(style);

            const status = document.createElement('p');
            status.id = 'ghonchu-status';
            status.textContent = '🔒 Activating security boundary and initializing...';
            status.style = 'font-size: 14px; color: #94a3b8; font-weight: bold;';

            // Privacy face masking toggle control
            const maskLabel = document.createElement('label');
            maskLabel.style = 'display: block; margin: 8px 0; font-size: 13px; color: #cbd5e1; cursor: pointer;';
            const maskCheckbox = document.createElement('input');
            maskCheckbox.type = 'checkbox';
            maskCheckbox.checked = true;
            maskCheckbox.style = 'margin-right: 6px;';
            maskLabel.appendChild(maskCheckbox);
            maskLabel.appendChild(document.createTextNode('Mask Human Faces (Optional - Default ON)'));

            // Hidden video helper to capture stream, display rendered canvas instead
            const video = document.createElement('video');
            video.style = 'display: none;';
            video.autoplay = true;
            video.playsInline = true;

            const viewCanvas = document.createElement('canvas');
            viewCanvas.style = 'width: 100%; border-radius: 6px; background: #020617; margin-bottom: 10px;';
            const viewCtx = viewCanvas.getContext('2d');

            // Web Speech Enable Button
            const voiceBtn = document.createElement('button');
            voiceBtn.textContent = '🎙️ Click to Enable Voice Interaction';
            voiceBtn.style = 'width: 100%; padding: 10px; margin-bottom: 8px; background: #d97706; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; transition: 0.3s;';

            const speechStatus = document.createElement('div');
            speechStatus.style = 'font-size: 13px; color: #fbbf24; margin: 8px 0; font-style: italic; min-height: 18px;';
            speechStatus.textContent = 'Permission prompt pending. Tap "Enable Voice" above.';

            // Fail-safe Text Input Block (Bypasses Web Speech sandbox limitations)
            const textInputContainer = document.createElement('div');
            textInputContainer.style = 'margin: 10px 0; display: flex; gap: 8px;';

            const textInput = document.createElement('input');
            textInput.type = 'text';
            textInput.placeholder = 'Type custom query for Ghonchu...';
            textInput.style = 'flex-grow: 1; padding: 8px; border-radius: 4px; border: 1px solid #78716c; background: #292524; color: white; font-size: 13px;';

            const sendTextBtn = document.createElement('button');
            sendTextBtn.textContent = 'Send';
            sendTextBtn.style = 'padding: 8px 14px; background: #16a34a; border: none; border-radius: 4px; color: white; font-weight: bold; cursor: pointer;';

            textInputContainer.appendChild(textInput);
            textInputContainer.appendChild(sendTextBtn);
            container.appendChild(textInputContainer);

            const stopBtn = document.createElement('button');
            stopBtn.textContent = 'Freeze Observation';
            stopBtn.style = 'width: 100%; padding: 12px; background: #78716c; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer;';

            container.appendChild(title);
            container.appendChild(mascotContainer);
            container.appendChild(status);
            container.appendChild(maskLabel);
            container.appendChild(viewCanvas);
            container.appendChild(voiceBtn);
            container.appendChild(speechStatus);
            container.appendChild(textInputContainer);
            container.appendChild(stopBtn);
            document.body.appendChild(container);

            const synth = window.speechSynthesis;
            let speakingActive = false;

            function speakFeedback(text) {
                // Fully flush previous utterance states to prevent browser freeze-ups
                synth.cancel();
                setTimeout(() => {
                    const utterance = new SpeechSynthesisUtterance(text);
                    utterance.rate = 0.95;
                    utterance.onstart = () => {
                        speakingActive = true;
                        mascotContainer.classList.add('talking-active');
                    };
                    utterance.onend = () => {
                        speakingActive = false;
                        mascotContainer.classList.remove('talking-active');
                    };
                    utterance.onerror = () => {
                        speakingActive = false;
                        mascotContainer.classList.remove('talking-active');
                    };
                    synth.speak(utterance);
                }, 50);}

            // Welcome speech introducing the Google Drive optional caching parameter
            speakFeedback("Greetings! I am Ghonchu. Point your camera at any historical artifact to study written lineages. If you wish to make my initialization blazing fast on future loads, you can check the optional Google Drive cache checkbox in the cell form controls. Let's research some history!");
            status.textContent = '👀 Actively scanning inscription surface...';

            const stream = await navigator.mediaDevices.getUserMedia({
                video: { facingMode: "environment" }
            }).catch(async (err) => {
                return await navigator.mediaDevices.getUserMedia({ video: true });
            });

            video.srcObject = stream;

            // Web Speech API configuration bound to user activation event
            const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
            let recognition;
            let voicePrompt = "";
            let recognitionActive = false;

            if (SpeechRecognition) {
                recognition = new SpeechRecognition();
                recognition.continuous = true;
                recognition.interimResults = false;
                recognition.lang = 'en-US';

                recognition.onresult = (event) => {
                    const queryResult = event.results[event.results.length - 1];
                    if (queryResult.isFinal) {
                        voicePrompt = queryResult[0].transcript.trim();
                        speechStatus.textContent = `🎙| Live prompt received. Analyzing...`;
                    }
                };

                recognition.onerror = (event) => {
                    console.error('Speech recognition error:', event.error);
                    if (event.error === 'not-allowed') {
                        speechStatus.textContent = '⚠️ Microphone blocked. Check browser address bar permissions or use the text box below!';
                        voiceBtn.style.backgroundColor = '#ef4444';
                        voiceBtn.textContent = '⚠️ Mic Access Denied';
                        recognitionActive = false;
                    }
                };

                recognition.onend = () => {
                    // Auto-restart recognition safely
                    if (activeLoop && recognitionActive) {
                        try {
                            recognition.start();
                        } catch(e) {}
                    }
                };

                voiceBtn.onclick = async () => {
                    if (!recognitionActive) {
                        try {
                            await navigator.mediaDevices.getUserMedia({ audio: true });
                            recognitionActive = true;
                            recognition.start();
                            speechStatus.textContent = '🎙| Assistant is actively listening... Speak now!';
                            voiceBtn.style.backgroundColor = '#16a34a';
                            voiceBtn.textContent = '🎙| Voice Active';
                        } catch (err) {
                            speechStatus.textContent = '⚠️ Microphone permission rejected or blocked. Please type in the box instead.';
                            voiceBtn.style.backgroundColor = '#ef4444';
                        }
                    } else {
                        recognitionActive = false;
                        try { recognition.stop(); } catch(e) {}
                        speechStatus.textContent = '🎙| Voice Assistant disabled.';
                        voiceBtn.style.backgroundColor = '#d97706';
                        voiceBtn.textContent = '🎙| Click to Enable Voice Interaction';
                    }
                };
            } else {
                speechStatus.textContent = '⚠️ Speech Recognition not supported in this browser. Please use the manual text entry.';
                voiceBtn.style.display = 'none';
            }

            // Interrupt speech when user types in the text box
            textInput.oninput = () => {
                synth.cancel();
            };

            // Bind Text Input Send click
            sendTextBtn.onclick = () => {
                synth.cancel();
                const typedText = textInput.value.trim();
                if (typedText) {
                    voicePrompt = typedText;
                    speechStatus.textContent = `💬 Typed prompt received. Analyzing...`;
                    textInput.value = '';
                }
            };

            const canvas = document.createElement('canvas');
            const ctx = canvas.getContext('2d');
            let activeLoop = true;

            stopBtn.onclick = () => {
                activeLoop = false;
                recognitionActive = false;
                if (recognition) {
                    try { recognition.stop(); } catch(e) {}
                }
                stream.getVideoTracks().forEach(track => track.stop());
                container.remove();
                speakFeedback("Observation session frozen.");
            };

            // Live canvas drawing and dynamic interactive processing loop
            async function captureAndSendLoop() {
                while (activeLoop) {
                    if (video.videoWidth > 0) {
                        viewCanvas.width = 448;
                        viewCanvas.height = (video.videoHeight / video.videoWidth) * 448;
                        viewCtx.drawImage(video, 0, 0, viewCanvas.width, viewCanvas.height);

                        canvas.width = viewCanvas.width;
                        canvas.height = viewCanvas.height;
                        ctx.drawImage(viewCanvas, 0, 0);

                        const hasPrompt = (voicePrompt.trim().length > 0);
                        const runMasking = maskCheckbox.checked;

                        // On-demand processing when there is a prompt present
                        if (hasPrompt) {
                            const frameData = canvas.toDataURL('image/jpeg', 0.8);

                            const result = await google.colab.kernel.invokeFunction(
                                'notebook.process_field_frame',
                                [frameData, runMasking, voicePrompt],
                                {}
                            );

                            voicePrompt = "";
                            const jsonResponse = result.data['application/json'];

                            if (jsonResponse && jsonResponse.voice_instruction) {
                                status.textContent = `🏛️ Ghonchu: ${jsonResponse.voice_instruction}`;
                                speakFeedback(jsonResponse.voice_instruction);
                            }
                        }
                    }
                    await new Promise(r => setTimeout(r, 1000));
                }
            }

            // Separate loop to update the UI frame with face masking at 20fps
            async function drawVideoLoop() {
                while (activeLoop) {
                    if (video.videoWidth > 0) {
                        viewCanvas.width = 448;
                        viewCanvas.height = (video.videoHeight / video.videoWidth) * 448;
                        viewCtx.drawImage(video, 0, 0, viewCanvas.width, viewCanvas.height);

                        if (maskCheckbox.checked) {
                            const tempCanvas = document.createElement('canvas');
                            tempCanvas.width = viewCanvas.width;
                            tempCanvas.height = viewCanvas.height;
                            tempCanvas.getContext('2d').drawImage(viewCanvas, 0, 0);

                            const frameData = tempCanvas.toDataURL('image/jpeg', 0.7);

                            const result = await google.colab.kernel.invokeFunction(
                                'notebook.process_masking_only',
                                [frameData],
                                {}
                            );
                            const jsonResponse = result.data['application/json'];
                            if (jsonResponse && jsonResponse.faces) {
                                jsonResponse.faces.forEach(([x, y, w, h]) => {
                                    viewCtx.fillStyle = 'black';
                                    viewCtx.fillRect(x, y, w, h);
                                });
                            }
                        }
                    }
                    await new Promise(r => setTimeout(r, 50));
                }
            }

            captureAndSendLoop();
            drawVideoLoop();
        }
        runGhonchuTutor();
    ''')
    display(js_code)

def get_web_search_summary(query):
    """Fallback search/summarization mechanism using duckduckgo html search"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        encoded_query = urllib.parse.quote(query)
        url = f"https://html.duckduckgo.com/html/?q={encoded_query}"
        res = requests.get(url, headers=headers, timeout=5)
        if res.status_code == 200:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(res.text, 'html.parser')
            snippets = [snippet.get_text().strip() for snippet in soup.find_all('a', class_='result__snippet')[:3]]
            if snippets:
                return " ".join(snippets)
    except Exception as e:
        pass
    return "No live search results available."

def process_masking_only(base64_frame):
    try:
        header, encoded = base64_frame.split(',')
        image_bytes = base64.b64decode(encoded)
        np_arr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 4)
        faces_list = [[int(x), int(y), int(w), int(h)] for (x, y, w, h) in faces]
        return IPython.display.JSON({"faces": faces_list})
    except Exception as e:
        return IPython.display.JSON({"faces": []})

def process_ghonchu_frame(base64_frame, should_mask, user_voice_prompt):
    try:
        # Check if the query is non-visual or conversational/general knowledge
        non_visual_keywords = ["what is your name", "who are you", "how are you", "weather", "capital of", "distance to", "definition of", "who written", "who built", "search for", "zistgah"]
        lower_query = user_voice_prompt.lower().strip() if user_voice_prompt else ""

        # Custom explicit response for Zistgah queries
        if "zistgah" in lower_query:
            voice_out = "Zistgah is an innovative scientific platform focused on biology, computational genetics, and interactive AI tutoring tools."
            log_and_stamp_artifact(user_voice_prompt, voice_out, None)
            return IPython.display.JSON({"voice_instruction": voice_out})

        # Exact match or general knowledge triggers
        if lower_query and (any(kw in lower_query for kw in non_visual_keywords) or len(lower_query.split()) < 3):
            if "name" in lower_query or "who are you" in lower_query:
                voice_out = "My name is Ghonchu, and I am your friendly, round, smiley-ball Field Epigraphy and History Assistant!"
                log_and_stamp_artifact(user_voice_prompt, voice_out, None)
                return IPython.display.JSON({"voice_instruction": voice_out})

            # Execute Web Search & Summarize Fallback
            summary_context = get_web_search_summary(user_voice_prompt)
            prompt_text = f"You are Ghonchu, the smiley-ball history assistant. The user asked: '{user_voice_prompt}'. Here is some live web information to synthesize: '{summary_context}'. Answer this question directly in exactly one short, informative sentence."

            messages = [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": prompt_text}]
                }
            ]
            input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
            inputs = tokenizer(None, input_text, return_tensors="pt").to("cuda")

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    temperature=0.2,
                    do_sample=True
                )
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            voice_instruction = generated_text.split("assistant\n")[-1].strip()
            log_and_stamp_artifact(user_voice_prompt, voice_instruction, None)
            return IPython.display.JSON({"voice_instruction": voice_instruction})

        # If there's no active user prompt, do not run the vision model output generation
        # to stop it from randomly describing the image repeatedly without user queries.
        if not user_voice_prompt:
            return IPython.display.JSON({"voice_instruction": ""})

        header, encoded = base64_frame.split(',')
        image_bytes = base64.b64decode(encoded)

        # Optional face masking using Haar Cascade
        if should_mask:
            np_arr = np.frombuffer(image_bytes, np.uint8)
            img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)
            for (x, y, w, h) in faces:
                cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 0), -1)
            _, encoded_masked = cv2.imencode('.jpg', img)
            pil_image = Image.open(BytesIO(encoded_masked.tobytes())).convert("RGB")
        else:
            pil_image = Image.open(BytesIO(image_bytes)).convert("RGB")

        prompt_text = FIELD_SYSTEM_PROMPT
        prompt_text += f"\nUSER EXPLICIT QUESTION: \"{user_voice_prompt}\". Do NOT repeat the question or parrot my words. Answer this question in exactly one sentence based solely on your analysis of this image."

        # 4. Formulate the Dynamic Context Chat Prompt targeting the local VLM layers
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_text},
                    {"type": "image"},
                    {"type": "text", "text": "Analyze the image. Answer the user query directly in a single informative sentence. Do not say 'You asked me...' or repeat the prompt."}
                ]
            }
        ]

        # Tokenize and pass variables directly to local GPU registers
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(pil_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=60,
                temperature=0.2,
                do_sample=True
            )

        # Extract the textual response slice cleanly
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Parse output down to clean assistant payload content string
        voice_instruction = generated_text.split("assistant\n")[-1].strip()

        # Save logs and OTS stamp
        log_and_stamp_artifact(user_voice_prompt, voice_instruction, pil_image)

        return IPython.display.JSON({"voice_instruction": voice_instruction})

    except Exception as e:
        return IPython.display.JSON({"voice_instruction": ""})

# Bind context callbacks
google.colab.output.register_callback('notebook.process_field_frame', process_ghonchu_frame)
google.colab.output.register_callback('notebook.process_masking_only', process_masking_only)

# Launch
start_ghonchu_eye_tutor()

Mounting Google Drive for caching...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Google Drive cache directory: /content/drive/MyDrive/ghonchu_model_cache
Using Google Drive OTS logs directory: /content/drive/MyDrive/ghonchu_ots_logs
Loading Ghonchu Vision Brain (Qwen2.5-VL-7B 4-bit)...
==((====))==  Unsloth 2026.11.3: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<IPython.core.display.Javascript object>

[OTS Active] Saved log and notarized snapshot to Google Drive.
[OTS Active] Saved log and notarized snapshot to Google Drive.
